# Cross-Market Comparison: Houston · Austin · SF · NYC

**Goal:** Compare neighborhood-level forecasts across four major US markets using H3 hex cells.

This notebook shows how to use `client.forecast.by_hex.retrieve_many()` to pull proforma metrics
for representative neighborhoods in each city and generate a side-by-side comparison.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/homecastr/homecastr-cookbook/blob/main/examples/neighborhoods/02_market_comparison.ipynb)

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np
from homecastr import HomecastrClient

client = HomecastrClient(os.getenv("HOMECASTR_API_KEY", "hc_demo_public_readonly"))

In [ ]:
# Representative H3 hex cells (resolution 8) for neighborhoods in each city.
# To find hex IDs for other areas, use: https://wolf-h3-viewer.glitch.me/
# or convert lat/lng with: h3.latlng_to_cell(lat, lng, 8)

MARKETS = {
    "Houston (Midtown)": "882a100c65fffff",
    "Houston (Heights)": "882a1008b5fffff",
    "Austin (East)": "8844c0a5b5fffff",
    "Austin (Domain)": "8844c0a4b5fffff",
    "San Francisco (Mission)": "8828308d95fffff",
    "San Francisco (SOMA)": "882830859dfffff",
    "New York (Park Slope)": "882a1072c5fffff",
    "New York (Astoria)": "882a107391fffff",
}

hex_ids = list(MARKETS.values())
labels = list(MARKETS.keys())

print(f"Fetching forecasts for {len(hex_ids)} neighborhoods...")

In [ ]:
df = client.forecast.by_hex.retrieve_many(hex_ids, year=2028)
df["market"] = labels
df["city"] = [m.split("(")[0].strip() for m in labels]
df.head()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Cross-Market Comparison: 2028 Homecastr Forecasts", fontsize=14, fontweight="bold")

colors = ["steelblue", "steelblue", "darkorange", "darkorange",
          "forestgreen", "forestgreen", "crimson", "crimson"]

# Median forecast value
axes[0, 0].barh(df["market"], df["p50"] / 1e3, color=colors)
axes[0, 0].set_xlabel("Median Forecast Value ($000s)")
axes[0, 0].set_title("P50 Forecast")

# Opportunity (appreciation)
axes[0, 1].barh(df["market"], df["opportunity"], color=colors)
axes[0, 1].axvline(0, color="k", linewidth=0.5)
axes[0, 1].set_xlabel("Appreciation (%)")
axes[0, 1].set_title("5-Year Opportunity")

# Cap rate
axes[1, 0].barh(df["market"], df["cap_rate"] * 100, color=colors)
axes[1, 0].set_xlabel("Cap Rate (%)")
axes[1, 0].set_title("Cap Rate")

# DSCR
axes[1, 1].barh(df["market"], df["dscr"], color=colors)
axes[1, 1].axvline(1.0, color="red", linewidth=1, linestyle="--", label="Break-even")
axes[1, 1].set_xlabel("DSCR")
axes[1, 1].set_title("Debt Service Coverage Ratio")
axes[1, 1].legend()

# City legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor="steelblue", label="Houston"),
    Patch(facecolor="darkorange", label="Austin"),
    Patch(facecolor="forestgreen", label="San Francisco"),
    Patch(facecolor="crimson", label="New York"),
]
fig.legend(handles=legend_elements, loc="lower center", ncol=4, frameon=False)

plt.tight_layout(rect=[0, 0.04, 1, 1])
plt.show()

In [ ]:
summary = df[["market", "predicted_value", "opportunity", "cap_rate", "dscr", "monthly_rent", "reliability"]].copy()
summary.columns = ["Market", "Median Value", "Appreciation %", "Cap Rate", "DSCR", "Monthly Rent", "Reliability"]
summary["Median Value"] = summary["Median Value"].apply(lambda x: f"${x:,.0f}" if pd.notna(x) else "—")
summary["Monthly Rent"] = summary["Monthly Rent"].apply(lambda x: f"${x:,.0f}" if pd.notna(x) else "—")
summary["Cap Rate"] = summary["Cap Rate"].apply(lambda x: f"{x*100:.2f}%" if pd.notna(x) else "—")
summary["DSCR"] = summary["DSCR"].apply(lambda x: f"{x:.2f}" if pd.notna(x) else "—")
summary["Reliability"] = summary["Reliability"].apply(lambda x: f"{x:.2f}" if pd.notna(x) else "—")
print(summary.to_string(index=False))